# Maternal Health Risk Stratification – Complete Analysis (Q1–Q8)

**StudyBuild Project 01 – Health & Medical Data Science Track**

This notebook performs a full, reproducible analysis of the UCI Maternal Health Risk dataset.
It covers data-quality checks, exploratory analysis, baseline modelling, error inspection and clinical interpretation.

**Important disclaimer:** This is an educational risk-stratification exercise, **not** a diagnostic tool.

## 0. Imports & Configuration

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
RANDOM_STATE = 42

## Q1 – What does the maternal-risk population look like?

### Data loading, missing-value check, duplicates and implausible values

In [ ]:
df = pd.read_csv("../data/Maternal Health Risk Data Set.csv")

print("=" * 60)
print("1. BASIC DATASET INFORMATION")
print("=" * 60)
print(f"Shape (rows, columns): {df.shape}")
print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nFirst 5 rows:\n{df.head()}")

print("\n" + "=" * 60)
print("2. MISSING VALUES CHECK")
print("=" * 60)
missing = df.isnull().sum()
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

print("\n" + "=" * 60)
print("3. DUPLICATE ROWS")
print("=" * 60)
n_duplicates = df.duplicated().sum()
print(f"Number of fully duplicated rows: {n_duplicates}")
print(f"Percentage of duplicates: {n_duplicates / len(df) * 100:.1f}%")

print("\n" + "=" * 60)
print("4. ORIGINAL CLASS DISTRIBUTION")
print("=" * 60)
print(df["RiskLevel"].value_counts())
print("\nNormalized:")
print(df["RiskLevel"].value_counts(normalize=True).round(3))

print("\n" + "=" * 60)
print("5. IMPLAUSIBLE / UNUSUAL VALUES")
print("=" * 60)
print("\nRecords with HeartRate < 40:")
print(df[df["HeartRate"] < 40])
print(f"\nAge < 15 or Age > 50: {(df['Age'] < 15).sum() + (df['Age'] > 50).sum()} records")
print(f"SystolicBP outside 70-180: {((df['SystolicBP'] < 70) | (df['SystolicBP'] > 180)).sum()}")
print(f"DiastolicBP outside 40-120: {((df['DiastolicBP'] < 40) | (df['DiastolicBP'] > 120)).sum()}")
print(f"BodyTemp outside 95-104 °F: {((df['BodyTemp'] < 95) | (df['BodyTemp'] > 104)).sum()}")
print(f"BS outside 4-25 mmol/L: {((df['BS'] < 4) | (df['BS'] > 25)).sum()}")

### Cleaning decisions

- **Version A**: remove only the two HeartRate = 7 records.
- **Version B (recommended)**: remove all exact duplicates **and** the two HeartRate = 7 records → **451 rows**.

Rationale: exact duplicates artificially inflate sample size and can produce optimistic performance estimates.

In [ ]:
df_version_A = df[df["HeartRate"] >= 40].copy().reset_index(drop=True)
df_clean = df.drop_duplicates().reset_index(drop=True)
df_clean = df_clean[df_clean["HeartRate"] >= 40].reset_index(drop=True)
df_clean["RiskLevel"] = df_clean["RiskLevel"].str.strip().str.lower()

print("=" * 60)
print("6. CLEANED VERSIONS")
print("=" * 60)
print(f"Version A (only HeartRate >= 40): {df_version_A.shape}")
print(f"Version B (drop duplicates + HeartRate >= 40): {df_clean.shape}")
print("\nClass distribution – Version B (recommended):")
print(df_clean["RiskLevel"].value_counts())
print("\nNormalized:")
print(df_clean["RiskLevel"].value_counts(normalize=True).round(3))

print("\n" + "=" * 60)
print("7. SUMMARY STATISTICS (Version B – recommended)")
print("=" * 60)
print(df_clean.describe().round(2))

df_clean.to_csv("../data/maternal_health_clean.csv", index=False)
print("\nCleaned file saved.")

### Q1 Key Findings

- Original file: 1 014 rows, 0 missing values, 562 exact duplicates (55.4 %).
- Two physiologically impossible HeartRate = 7 records removed.
- Cleaned dataset (Version B): **451 unique records**.
- Class balance after cleaning: Low risk 51.7 %, High risk 24.8 %, Mid risk 23.5 %.
- Extreme ages (Age < 15 or > 50) retained but flagged as unusual for pregnancy.

## Q2 – Which measurements differ most across risk groups?

In [ ]:
features = ["Age", "SystolicBP", "DiastolicBP", "BS", "BodyTemp", "HeartRate"]
risk_order = ["low risk", "mid risk", "high risk"]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feature in enumerate(features):
    sns.boxplot(
        x="RiskLevel", y=feature, data=df_clean,
        order=risk_order, ax=axes[i], palette="Set2"
    )
    axes[i].set_title(f"Distribution of {feature} by Risk Level", fontsize=12, fontweight="bold")
    axes[i].set_xlabel("Risk Level")
    axes[i].set_ylabel(feature)

plt.tight_layout()
plt.savefig("../figures/q2_risk_feature_distributions.png", dpi=300)
plt.show()

q2_summary = df_clean.groupby("RiskLevel")[features].agg(["mean", "median", "std"]).round(2)
print("=== Q2 GROUP SUMMARIES ===")
print(q2_summary)

### Q2 Interpretation

Box-plots show that **Blood Sugar (BS)** and **SystolicBP** produce the clearest separation:
- High-risk median BS ≈ 11 mmol/L versus ≈ 7 mmol/L in Low/Mid groups.
- SystolicBP is systematically higher in the High-risk group.
- HeartRate and BodyTemp exhibit substantial overlap across all three classes.

## Q3 – Which variables appear most strongly associated with High Risk?

In [ ]:
df_clean["IsHighRisk"] = (df_clean["RiskLevel"] == "high risk").astype(int)

corr_table = pd.DataFrame({
    "Pearson_Corr": df_clean[features].apply(lambda x: x.corr(df_clean["IsHighRisk"])),
    "Spearman_Corr": df_clean[features].apply(lambda x: x.corr(df_clean["IsHighRisk"], method="spearman"))
}).sort_values(by="Pearson_Corr", ascending=False)

print("=== Q3 CORRELATION WITH HIGH RISK ===")
print(corr_table.round(3))

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_clean, x="SystolicBP", y="BS", hue="RiskLevel",
    palette={"low risk": "#2ecc71", "mid risk": "#f39c12", "high risk": "#e74c3c"},
    alpha=0.7, s=70
)
plt.title("Blood Sugar (BS) vs Systolic Blood Pressure by Risk Level", fontsize=12, fontweight="bold")
plt.xlabel("Systolic Blood Pressure (mmHg)")
plt.ylabel("Blood Sugar (mmol/L)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(title="Risk Level")
plt.tight_layout()
plt.savefig("../figures/q3_scatter_bs_vs_sysbp.png", dpi=300)
plt.show()

### Q3 Interpretation & Causation Note

| Feature      | Pearson r | Rank |
|--------------|-----------|------|
| BS           | 0.573     | 1    |
| SystolicBP   | 0.288     | 2    |
| DiastolicBP  | 0.256     | 3    |
| BodyTemp     | 0.218     | 4    |
| Age          | 0.188     | 5    |
| HeartRate    | 0.182     | 6    |

**Association does not equal causation.** Elevated blood sugar is a strong *marker* of metabolic disturbance (e.g. gestational diabetes) that increases the probability of adverse outcomes; it is not claimed to be the sole causal factor.

## Q4 – Can a simple baseline model classify maternal risk?

In [ ]:
X = df_clean[features]
y = df_clean["RiskLevel"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)

dt_model = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

print("=== LOGISTIC REGRESSION ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(classification_report(y_test, y_pred_lr))

print("\n=== DECISION TREE (max_depth=4) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(classification_report(y_test, y_pred_dt))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
labels = ["high risk", "low risk", "mid risk"]

sns.heatmap(confusion_matrix(y_test, y_pred_lr, labels=labels),
            annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title("Logistic Regression Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

sns.heatmap(confusion_matrix(y_test, y_pred_dt, labels=labels),
            annot=True, fmt="d", cmap="Greens",
            xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title("Decision Tree (max_depth=4) Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.savefig("../figures/q4_confusion_matrices.png", dpi=300)
plt.show()

### Q4 Model Choice

A shallow Decision Tree (max_depth=4) was selected as the primary baseline because:
- it is fully interpretable,
- it does not require feature scaling,
- it slightly outperformed Logistic Regression on overall accuracy (68.1 % vs 67.0 %).

No complex ensembles were used, in accordance with project guidelines.

## Q5 – Is overall accuracy enough?

In [ ]:
print("=== Q5 – ACCURACY IS NOT ENOUGH ===")
print(f"Overall Accuracy (Decision Tree): {accuracy_score(y_test, y_pred_dt):.4f}\n")
print("Class-wise Metrics:")
print(classification_report(y_test, y_pred_dt))

cm = confusion_matrix(y_test, y_pred_dt, labels=labels)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds",
            xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix – Detailed Class Errors", fontsize=12, fontweight="bold")
plt.xlabel("Predicted Risk Level")
plt.ylabel("Actual Risk Level")
plt.tight_layout()
plt.savefig("../figures/q5_confusion_matrix_detailed.png", dpi=300)
plt.show()

### Q5 Clinical Evaluation

| Class      | Precision | Recall | F1-Score | Support |
|------------|-----------|--------|----------|---------|
| high risk  | 0.93      | **0.61** | 0.74   | 23      |
| low risk   | 0.64      | 0.98   | 0.77     | 47      |
| mid risk   | 0.50      | **0.10** | 0.16   | 21      |

**Overall accuracy (68.1 %) is misleading.**  
Missing a true High-risk case (False Negative) is clinically far more serious than other error types. The model correctly identifies only 61 % of High-risk patients and almost completely fails on the Mid-risk class.

## Q6 – Where does the model fail?

In [ ]:
test_analysis = X_test.copy()
test_analysis["Actual"] = y_test.values
test_analysis["Predicted"] = y_pred_dt
test_analysis["Is_Error"] = test_analysis["Actual"] != test_analysis["Predicted"]

misclassified = test_analysis[test_analysis["Is_Error"]]
error_pairs = (
    misclassified.groupby(["Actual", "Predicted"])
    .size()
    .reset_index(name="Count")
)
error_pairs["Error_Share_%"] = (error_pairs["Count"] / len(misclassified) * 100).round(2)
error_pairs = error_pairs.sort_values(by="Count", ascending=False)

print("=== Q6 MISCLASSIFICATION PAIRS ===")
print(error_pairs.to_string(index=False))

plt.figure(figsize=(8, 4.5))
sns.barplot(
    data=error_pairs,
    x="Count",
    y=error_pairs.apply(lambda r: f"{r['Actual'].title()} → {r['Predicted'].title()}", axis=1),
    palette="Reds_r"
)
plt.title("Q6: Distribution of Misclassification Errors", fontsize=12, fontweight="bold")
plt.xlabel("Number of Misclassified Test Cases")
plt.ylabel("Error Pair (Actual → Predicted)")
plt.tight_layout()
plt.savefig("../figures/q6_error_distribution.png", dpi=300)
plt.show()

### Q6 Error Analysis

| Actual     | Predicted | Count | Share  |
|------------|-----------|-------|--------|
| Mid risk   | Low risk  | 18    | 62.1 % |
| High risk  | Low risk  | 8     | 27.6 % |
| High risk  | Mid risk  | 1     | 3.5 %  |
| Low risk   | Mid risk  | 1     | 3.5 %  |
| Mid risk   | High risk | 1     | 3.5 %  |

The dominant failure mode is **Mid → Low**, reflecting heavy feature-space overlap. The second most frequent (and clinically most dangerous) error is **High → Low**.

## Q7 – Which features drive the model?

In [ ]:
importances = pd.DataFrame({
    "Feature": X.columns,
    "Importance": dt_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("=== Q7 FEATURE IMPORTANCES ===")
print(importances.to_string(index=False))

plt.figure(figsize=(8, 4.5))
sns.barplot(data=importances, x="Importance", y="Feature", palette="Blues_r")
plt.title("Q7: Feature Importance (Gini Impurity Decrease)", fontsize=12, fontweight="bold")
plt.xlabel("Gini Importance Score")
plt.ylabel("Clinical Feature")
plt.tight_layout()
plt.savefig("../figures/q7_feature_importance.png", dpi=300)
plt.show()

### Q7 Interpretation

| Feature     | Gini Importance | Role                                      |
|-------------|-----------------|-------------------------------------------|
| BS          | 56.8 %          | Primary split criterion                   |
| SystolicBP  | 26.4 %          | Secondary vascular risk separator         |
| BodyTemp    | 10.2 %          | Captures febrile cases                    |
| Age         | 6.7 %           | Minor contribution                        |
| DiastolicBP | 0.0 %           | Unused by the tree                        |
| HeartRate   | 0.0 %           | Unused by the tree                        |

Feature importance reflects **association inside this particular model**, not clinical causation.

## Q8 – What should a clinician or decision-maker take from this analysis?

In [ ]:
print("=" * 60)
print("Q8 – EXECUTIVE SUMMARY & CLINICAL IMPLICATIONS")
print("=" * 60)

summary_text = f"""
1. KEY FINDINGS
   - Primary clinical biomarkers: BS (56.8 % importance) and SystolicBP (26.4 %).
   - Decision Tree overall accuracy on held-out test set (n = {len(y_test)}): {accuracy_score(y_test, y_pred_dt)*100:.1f} %.

2. CLINICAL SAFETY
   - High-risk Recall = 0.61 → approximately 4 out of 10 true high-risk cases are missed.
   - The model must never be used as a stand-alone diagnostic tool.

3. LIMITATIONS
   - Original data contained 55.4 % exact duplicates.
   - Small cleaned sample (n = 451) and single geographic origin (rural Bangladesh).
   - Important covariates missing: proteinuria, gestational age, BMI, parity, etc.

4. RECOMMENDATIONS
   - Use only as a preliminary screening aid under clinician supervision.
   - Collect additional clinical variables before any real-world deployment.
   - Prefer models that explicitly optimise High-risk Recall (or cost-sensitive learning).
"""
print(summary_text)

fig, ax = plt.subplots(figsize=(8, 4))
y_pos = np.arange(len(importances))
ax.barh(y_pos, importances["Importance"], align="center", color="teal", alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(importances["Feature"])
ax.invert_yaxis()
ax.set_xlabel("Relative Impact Score")
ax.set_title("Q8 Executive Overview: Key Clinical Biomarkers Order", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("../figures/q8_executive_summary_chart.png", dpi=300)
plt.show()

---

**End of analysis.**  
All figures are saved in the `figures/` folder.  
A full narrative PDF report is available in `report/summary.pdf`.